# 🛠️ Урок 22 — Модель и интерфейс (материалы преподавателя)

Улучшаем модель → честно оцениваем → Gradio → README и слайды.

> Предполагается, что первичная модель `model` с урока 21 уже есть.

## Шаг 0 · Восстановим модель с урока 21 (для примера)

In [ ]:
import seaborn as sns
from sklearn.model_selection import train_test_split
df = sns.load_dataset('titanic')          # 👈 замени на свой датасет
num = ['age','fare','sibsp','parch']       # 👈 свои числовые признаки
cat = ['sex','pclass','embarked']          # 👈 свои категориальные признаки
X = df[num+cat]; y = df['survived']         # 👈 свой target
X_tr,X_te,y_tr,y_te = train_test_split(X,y,test_size=0.2,random_state=42)
print('Строк:', len(X), '| Признаков:', X.shape[1])
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
prep = ColumnTransformer([
    ('num', Pipeline([('i',SimpleImputer(strategy='median')),('s',StandardScaler())]), num),
    ('cat', Pipeline([('i',SimpleImputer(strategy='most_frequent')),('o',OneHotEncoder(handle_unknown='ignore'))]), cat)])
model = Pipeline([('prep',prep),('rf',RandomForestClassifier(n_estimators=100,random_state=42))])
model.fit(X_tr, y_tr)
print(f'Первичная модель: {accuracy_score(y_te, model.predict(X_te)):.0%}')

## Шаг 1 · Улучшение + честная оценка
Подбор параметров (GridSearchCV), метрика под задачу, кросс-валидация.

In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import classification_report
grid = GridSearchCV(model, {'rf__n_estimators':[100,300], 'rf__max_depth':[None,5,10]}, cv=5)
grid.fit(X_tr, y_tr)
print('Лучшие параметры:', grid.best_params_)
best = grid.best_estimator_
print(classification_report(y_te, best.predict(X_te)))
print(f'Кросс-валидация: {cross_val_score(best, X, y, cv=5).mean():.1%}')

## Шаг 2 (в Colab) · Веб-интерфейс через Gradio

In [ ]:
!pip install gradio -q
import gradio as gr, pandas as pd
def predict(age, fare, sex, pclass):
    row = pd.DataFrame([{'age':age,'fare':fare,'sibsp':0,'parch':0,'sex':sex,'pclass':pclass,'embarked':'S'}])
    return 'выжил' if best.predict(row)[0]==1 else 'погиб'
gr.Interface(fn=predict,
    inputs=[gr.Number(label='Возраст'), gr.Number(label='Цена билета'),
            gr.Radio(['male','female'], label='Пол'), gr.Radio([1,2,3], label='Класс')],
    outputs=gr.Label(label='Прогноз'), title='Мой проект').launch(share=True)

## Шаг 3 · README и структура презентации
**README.md:**
```markdown
# Название проекта
## Задача
Что предсказываем, метрика.
## Данные
Источник, размер, признаки.
## Модель
Какая, в Pipeline, сравнение с baseline.
## Результаты
Метрика на test/CV, где ошибается.
## Как запустить
Colab / ссылка на приложение.
```

**Слайды (3–5):** задача → данные → модель и метрика → демо → выводы.

---
**Итог урока 22.** Модель улучшена и честно оценена, есть интерфейс, готова структура рассказа. Дальше (урок 23) — репетиция и финальный деплой.